# 03 — Train DeepScalper (Equities, TradeMaster-Aligned)
Trains `DeepScalperNet` in this repo with **TradeMaster-aligned DQN controls** for the current equities workflow.

- Source: `configs/algorithmic_trading/algorithmic_trading_BTC_deepscalper_deepscalper_adam_mse.py`
- Agent core: `gamma=0.9`, `repeat_times=1`, `batch_size=64`, `clip_grad_norm=3.0`, `soft_update_tau=0.0`, `state_value_tau=0.005`
- Trainer core: `epochs=20`, `horizon_len=128`, `buffer_size=1e6`
- Optimizer: `Adam(lr=0.001)`
- Exploration: `explore_rate=0.25` (static)

This notebook uses the local environment/model implementation while keeping **shared hyperparameter fields 1:1 with TradeMaster values**.

**Input:**  `/content/drive/MyDrive/algo_trader/data/raw/{SYMBOL}.parquet`
**Output:** `/content/drive/MyDrive/algo_trader/weights/{SYMBOL}.pth`

**Notes (current codepath):**
- Training uses the current default agent flow (`update_net`) with TradeMaster-style controls.
- Active loss path is Q-loss only in the current project path.

Enable GPU: **Runtime -> Change runtime type -> T4 GPU**

In [ ]:
!pip install -q torch torchvision gymnasium numpy pandas pyarrow pytz tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
RAW_DIR     = '/content/drive/MyDrive/algo_trader/data/raw'
WEIGHTS_DIR = '/content/drive/MyDrive/algo_trader/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print(f'Raw data dir:  {RAW_DIR}')
print(f'Weights dir:   {WEIGHTS_DIR}')

In [ ]:
REPO_URL = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'
REPO_DIR = '/content/deepscalper_copilot'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# algo_trader on path → enables `from colab.deepscalper.X import Y`
ALGO_DIR = REPO_DIR + '/algo_trader'
if ALGO_DIR not in sys.path:
    sys.path.insert(0, ALGO_DIR)
print('Repo on path ✓')

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass
    print('Enabled cuDNN benchmark + TF32 matmul for higher throughput.')

# -- Architecture -- keep local project architecture ---------------------------------
MACRO_DIM     = 11
LOB_DIM       = 5
PRIV_DIM      = 2
N_DIR         = 3
N_SIZE        = 4
GRU_HIDDEN    = 128
MACRO_EMBED   = 64
FC_HIDDEN     = 128
LOOKBACK_BARS = 60
HINDSIGHT_HORIZON = 60

# -- TradeMaster 1.0.0 DeepScalper hyperparameters ----------------------------------
# Source:
# configs/algorithmic_trading/algorithmic_trading_BTC_deepscalper_deepscalper_adam_mse.py
# configs/_base_/nets/deepscalper.py
TRAINING_CONFIG = dict(
    epochs                = 20,         # trainer.epochs
    lr                    = 1e-3,       # optimizer Adam lr
    gamma                 = 0.9,        # agent.gamma
    repeat_times          = 1.0,        # agent.repeat_times
    batch_size            = 64,         # agent/trainer.batch_size
    buffer_size           = 1_000_000,  # trainer.buffer_size
    horizon_len           = 128,        # trainer.horizon_len
    clip_grad_norm        = 3.0,        # agent.clip_grad_norm
    soft_update_tau       = 0.0,        # agent.soft_update_tau
    state_value_tau       = 0.005,      # agent.state_value_tau (compat field)
    explore_rate          = 0.25,       # act.explore_rate
    eval_every            = 1,
    eval_episodes         = 1,
    train_transaction_cost_pct = 0.001,  # AlgorithmicTradingEnvironment default
    hindsight_weight          = 0.2,      # dataset future_weights default
    freq_ctrl_activity_bonus  = 0.0,      # disabled to stay TradeMaster-like
    freq_ctrl_churn_penalty   = 0.0,      # disabled to stay TradeMaster-like
)

# Current project universe: equities pilot set
try:
    import os, sys
    ROOT_DIR = os.path.join(REPO_DIR, 'algo_trader')
    if ROOT_DIR not in sys.path:
        sys.path.insert(0, ROOT_DIR)
    from tickers import SP100_TICKERS
    TRADING_UNIVERSE = SP100_TICKERS[:10]
except Exception:
    TRADING_UNIVERSE = ['AAPL']

print(f'Trading symbols ({len(TRADING_UNIVERSE)}): {TRADING_UNIVERSE}')
print(f'Training config: {TRAINING_CONFIG}')

PyTorch 2.10.0+cpu
Device: cpu
Crypto pairs: ['BTC/USD']
Training config: {'n_episodes': 300, 'lr': 0.0001, 'tau': 0.005, 'buffer_capacity': 100000, 'batch_size': 1024, 'replay_prefill_steps': 8192, 'updates_per_step': 16, 'epsilon_decay': 20000, 'early_stop_patience': 30, 'eval_every': 10, 'eval_episodes': 10, 'gradient_clip': 0.5, 'dropout': 0.1}


In [ ]:
import subprocess

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
from colab.deepscalper.agent import DeepScalperAgent
from colab.deepscalper.environment import ScalperEnv
from colab.deepscalper.utils import (
    compute_macro_features,
    compute_micro_features,
    compute_day_starts,
)

_EQUITY_ANNUALISE = np.sqrt(98_280)


def compute_episode_sharpe(episode_log_returns: list) -> float:
    rets = np.asarray(episode_log_returns, dtype=np.float64)
    if len(rets) < 5:
        return 0.0
    std = rets.std()
    if std == 0:
        return 0.0
    return float((rets.mean() / std) * _EQUITY_ANNUALISE)


def evaluate_agent(agent: DeepScalperAgent, env: ScalperEnv, n_episodes: int) -> float:
    saved_explore_rate = agent.explore_rate
    agent.explore_rate = TRAINING_CONFIG['explore_rate']
    all_log_returns = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            dir_act, _size_act = agent.select_action(obs)
            obs, r, terminated, truncated, info = env.step(dir_act)
            all_log_returns.append(info.get('log_return', r))
            done = terminated or truncated
    agent.explore_rate = saved_explore_rate
    return compute_episode_sharpe(all_log_returns)


def _gpu_stats() -> dict:
    if DEVICE != 'cuda':
        return {'util': 0, 'mem_used_gb': 0.0, 'mem_total_gb': 0.0}

    try:
        result = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=utilization.gpu,memory.used,memory.total',
                '--format=csv,noheader,nounits',
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        util_str, used_str, total_str = result.stdout.strip().splitlines()[0].split(', ')
        return {
            'util': int(util_str),
            'mem_used_gb': float(used_str) / 1024.0,
            'mem_total_gb': float(total_str) / 1024.0,
        }
    except Exception:
        return {
            'util': -1,
            'mem_used_gb': float(torch.cuda.memory_allocated() / (1024 ** 3)),
            'mem_total_gb': float(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)),
        }


def _format_gpu_postfix() -> dict:
    stats = _gpu_stats()
    if DEVICE != 'cuda':
        return {}
    return {
        'gpu%': stats['util'],
        'vram': f"{stats['mem_used_gb']:.1f}/{stats['mem_total_gb']:.0f}GB",
    }


def shape_reward_with_trade_frequency_control(
    reward: float,
    position: int,
    trade_occurred: int,
    cfg: dict,
) -> float:
    activity_bonus = float(cfg['freq_ctrl_activity_bonus']) * float(position != 0)
    churn_penalty = float(cfg['freq_ctrl_churn_penalty']) * float(trade_occurred)
    return float(reward) + activity_bonus - churn_penalty


def prefill_replay(agent: DeepScalperAgent, env: ScalperEnv, min_steps: int) -> None:
    prefill_pbar = tqdm(total=min_steps, desc='Replay prefill', leave=False)
    steps = 0
    obs, _ = env.reset()
    prev_pos = 0

    while steps < min_steps:
        dir_action = int(env.action_space.sample())
        size_action = int(np.random.randint(N_SIZE))
        next_obs, reward, terminated, truncated, info = env.step(dir_action)
        done = terminated or truncated
        pos = int(info.get('position', 0))
        trade_occurred = int(pos != prev_pos)
        shaped_reward = shape_reward_with_trade_frequency_control(
            reward=float(reward),
            position=pos,
            trade_occurred=trade_occurred,
            cfg=TRAINING_CONFIG,
        )
        agent.store(
            obs=obs,
            dir_action=dir_action,
            size_action=size_action,
            reward=shaped_reward,
            next_obs=next_obs,
            done=done,
        )
        obs = next_obs
        prev_pos = pos
        steps += 1
        prefill_pbar.update(1)
        if steps % 128 == 0:
            prefill_pbar.set_postfix(_format_gpu_postfix())
        if done:
            obs, _ = env.reset()

    prefill_pbar.close()


training_log = []
for symbol in tqdm(TRADING_UNIVERSE, desc='Symbols'):
    safe_name = symbol.replace('/', '_')
    weights_path = f'{WEIGHTS_DIR}/{safe_name}.pth'

    if os.path.exists(weights_path):
        print(f'{symbol}: weights already exist - skipping.')
        continue

    raw_path = f'{RAW_DIR}/{safe_name}.parquet'
    if not os.path.exists(raw_path):
        print(f'WARNING: {raw_path} missing - run 01_fetch_training_data first.')
        continue

    bars = pd.read_parquet(raw_path)
    bars.columns = [c.lower() for c in bars.columns]
    bars = bars[['open', 'high', 'low', 'close', 'volume']].astype(float)

    macro_feats = compute_macro_features(bars)
    lob_feats = compute_micro_features(bars, use_proxy=True)
    close_arr = bars['close'].values.astype(np.float64)
    day_starts = compute_day_starts(bars.index)

    if len(day_starts) < 14:
        print(f'WARNING: {symbol} has only {len(day_starts)} days - need >=14, skipping.')
        continue

    n = len(bars)
    train_end = int(n * 0.70)
    val_end = int(n * 0.80)

    train_df = bars.iloc[:train_end]
    val_df = bars.iloc[train_end:val_end]
    test_df = bars.iloc[val_end:]

    assert train_df.index[-1] < val_df.index[0], 'Train/val overlap detected!'
    assert val_df.index[-1] < test_df.index[0], 'Val/test overlap detected!'

    train_day_starts = [d for d in day_starts if d < train_end]
    val_day_starts = [d - train_end for d in day_starts if train_end <= d < val_end]
    test_day_starts = [d - val_end for d in day_starts if d >= val_end]

    if not train_day_starts or not val_day_starts or not test_day_starts:
        print(f'WARNING: {symbol} insufficient days in one split - skipping.')
        continue

    train_env = ScalperEnv(
        lob_features=lob_feats[:train_end],
        macro_features=macro_feats[:train_end],
        close_prices=close_arr[:train_end],
        day_starts=train_day_starts,
        lookback_bars=LOOKBACK_BARS,
        transaction_cost_pct=TRAINING_CONFIG['train_transaction_cost_pct'],
        hindsight_horizon=HINDSIGHT_HORIZON,
        hindsight_weight=TRAINING_CONFIG['hindsight_weight'],
    )
    val_env = ScalperEnv(
        lob_features=lob_feats[train_end:val_end],
        macro_features=macro_feats[train_end:val_end],
        close_prices=close_arr[train_end:val_end],
        day_starts=val_day_starts,
        lookback_bars=LOOKBACK_BARS,
        hindsight_horizon=HINDSIGHT_HORIZON,
    )
    test_env = ScalperEnv(
        lob_features=lob_feats[val_end:],
        macro_features=macro_feats[val_end:],
        close_prices=close_arr[val_end:],
        day_starts=test_day_starts,
        lookback_bars=LOOKBACK_BARS,
        hindsight_horizon=HINDSIGHT_HORIZON,
    )

    resolved_batch_size = int(TRAINING_CONFIG['batch_size'])
    print(f'{symbol}: using TradeMaster-aligned batch_size={resolved_batch_size}')

    agent = DeepScalperAgent(
        macro_dim=MACRO_DIM,
        lob_dim=LOB_DIM,
        priv_dim=PRIV_DIM,
        n_dir=N_DIR,
        n_size=N_SIZE,
        gru_hidden=GRU_HIDDEN,
        macro_embed=MACRO_EMBED,
        fc_hidden=FC_HIDDEN,
        device=DEVICE,
        lr=TRAINING_CONFIG['lr'],
        gamma=TRAINING_CONFIG['gamma'],
        repeat_times=TRAINING_CONFIG['repeat_times'],
        clip_grad_norm=TRAINING_CONFIG['clip_grad_norm'],
        soft_update_tau=TRAINING_CONFIG['soft_update_tau'],
        state_value_tau=TRAINING_CONFIG['state_value_tau'],
        batch_size=resolved_batch_size,
        buffer_capacity=TRAINING_CONFIG['buffer_size'],
        explore_rate=TRAINING_CONFIG['explore_rate'],
    )

    prefill_steps = max(int(TRAINING_CONFIG['horizon_len']), resolved_batch_size)
    prefill_replay(agent, train_env, prefill_steps)

    best_sharpe = -np.inf
    max_episodes = TRAINING_CONFIG['epochs']
    eval_every = TRAINING_CONFIG['eval_every']
    eval_episodes = TRAINING_CONFIG['eval_episodes']

    episode_pbar = tqdm(range(1, max_episodes + 1), desc=f'{symbol} episodes', leave=False)
    for episode in episode_pbar:
        obs, _ = train_env.reset()
        done = False
        episode_reward = 0.0
        episode_returns = []
        learn_losses = []
        step_count = 0
        grad_updates = 0

        while not done:
            dir_act, size_act = agent.select_action(obs)
            next_obs, reward, terminated, truncated, info = train_env.step(dir_act)
            done = terminated or truncated
            prev_pos = int(obs['priv'][-1, 0])
            new_pos = int(info.get('position', prev_pos))
            trade_occurred = int(new_pos != prev_pos)
            shaped_reward = shape_reward_with_trade_frequency_control(
                reward=float(reward),
                position=new_pos,
                trade_occurred=trade_occurred,
                cfg=TRAINING_CONFIG,
            )

            agent.store(
                obs=obs,
                dir_action=dir_act,
                size_action=size_act,
                reward=shaped_reward,
                next_obs=next_obs,
                done=done,
            )

            loss = agent.update_net()
            if loss is not None:
                learn_losses.append(loss)
                grad_updates += max(1, int(TRAINING_CONFIG['repeat_times']))

            obs = next_obs
            episode_reward += float(shaped_reward)
            episode_returns.append(info.get('log_return', 0.0))
            step_count += 1

        mean_loss = float(np.mean(learn_losses)) if learn_losses else float('nan')
        episode_sharpe = compute_episode_sharpe(episode_returns)
        postfix = {
            'eps': f'{agent.explore_rate:.3f}',
            'steps': step_count,
            'upd': grad_updates,
            'rew': f'{episode_reward:.3f}',
            'shr': f'{episode_sharpe:.3f}',
            'loss': (f'{mean_loss:.4f}' if not np.isnan(mean_loss) else 'idle'),
        }
        postfix.update(_format_gpu_postfix())
        episode_pbar.set_postfix(postfix)
        agent.save(weights_path)

        if episode % eval_every == 0:
            val_sharpe = evaluate_agent(agent, val_env, eval_episodes)
            if val_sharpe > best_sharpe:
                best_sharpe = val_sharpe
                agent.save(weights_path)

            eval_postfix = {
                'eps': f'{agent.explore_rate:.3f}',
                'val': f'{val_sharpe:.3f}',
                'best': f'{best_sharpe:.3f}',
            }
            eval_postfix.update(_format_gpu_postfix())
            episode_pbar.set_postfix(eval_postfix)

    episode_pbar.close()

    if not os.path.exists(weights_path):
        agent.save(weights_path)

    print(f'\n{symbol}: Running out-of-sample test evaluation...')
    saved_explore_rate = agent.explore_rate
    agent.explore_rate = TRAINING_CONFIG['explore_rate']
    test_log_returns = []
    test_positions = []
    trade_durations = []
    current_trade_len = 0

    for _ in range(len(test_day_starts)):
        obs, _ = test_env.reset()
        done = False
        while not done:
            dir_act, _size_act = agent.select_action(obs)
            obs, r, terminated, truncated, info = test_env.step(dir_act)
            log_ret = info.get('log_return', 0.0)
            pos = info.get('position', 0)
            test_log_returns.append(log_ret)
            test_positions.append(pos)
            if pos != 0:
                current_trade_len += 1
            elif current_trade_len > 0:
                trade_durations.append(current_trade_len)
                current_trade_len = 0
            done = terminated or truncated

    if current_trade_len > 0:
        trade_durations.append(current_trade_len)

    agent.explore_rate = saved_explore_rate

    rets = np.array(test_log_returns)
    cum = np.cumsum(rets)
    peak = np.maximum.accumulate(cum)
    max_dd = float(np.abs(cum - peak).max())

    test_sharpe = compute_episode_sharpe(rets.tolist())
    win_rate = float(np.mean(rets > 0)) if len(rets) > 0 else 0.0
    avg_trade_dur = float(np.mean(trade_durations)) if trade_durations else 0.0

    print(f'  Test Sharpe:         {test_sharpe:.4f}  (target >= 0.5)')
    print(f'  Max Drawdown:        {max_dd*100:.2f}%   (limit <= 15%)')
    print(f'  Win Rate:            {win_rate*100:.1f}%     (target >= 45%)')
    print(f'  Avg Trade Duration:  {avg_trade_dur:.1f} bars  (target >= 3)')

    gate_passed = test_sharpe >= 0.5
    if not gate_passed:
        print(f'  [WARN] GATE FAILED: Sharpe {test_sharpe:.4f} < 0.5 - model NOT accepted.')
        print('      Run 06_sharpe_diagnosis.ipynb to diagnose and improve.')
    else:
        print(f'  [OK] All gates passed - weights saved to {weights_path}')

    training_log.append({
        'symbol': symbol,
        'best_val_sharpe': round(best_sharpe, 4),
        'test_sharpe': round(test_sharpe, 4),
        'max_drawdown_pct': round(max_dd * 100, 2),
        'win_rate_pct': round(win_rate * 100, 1),
        'avg_trade_duration': round(avg_trade_dur, 1),
        'gate_passed': gate_passed,
        'batch_size': resolved_batch_size,
        'repeat_times': TRAINING_CONFIG['repeat_times'],
        'horizon_len': TRAINING_CONFIG['horizon_len'],
        'prefill_steps': prefill_steps,
    })
    print(f'{symbol}: best val Sharpe = {best_sharpe:.4f}  |  test Sharpe = {test_sharpe:.4f}')

print('\n=== TRAINING COMPLETE ===')
if training_log:
    df_log = pd.DataFrame(training_log).sort_values('test_sharpe', ascending=False)
    print(df_log.to_string(index=False))

log_path = '/content/drive/MyDrive/algo_trader/training_log.csv'
pd.DataFrame(training_log).to_csv(log_path, index=False)
print(f'\nTraining log saved -> {log_path}')